In [62]:
import yfinance as yf
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.linear_model import LinearRegression

# =====================
# LISTA DE BOLSAS GLOBAIS COM TIPO
# =====================
tickers = {
    "^BVSP": {"nome": "Ibovespa (Brasil)", "tipo": "Emergente"},
    "^GSPC": {"nome": "S&P500 (EUA)", "tipo": "Desenvolvida"},
    "^IXIC": {"nome": "Nasdaq (EUA)", "tipo": "Desenvolvida"},
    "^GDAXI": {"nome": "DAX (Alemanha)", "tipo": "Desenvolvida"},
    "^FTSE": {"nome": "FTSE100 (Reino Unido)", "tipo": "Desenvolvida"},
    "^FCHI": {"nome": "CAC40 (França)", "tipo": "Desenvolvida"},
    "^N225": {"nome": "Nikkei 225 (Japão)", "tipo": "Desenvolvida"},
    "^HSI": {"nome": "Hang Seng (Hong Kong)", "tipo": "Emergente"},
    "000001.SS": {"nome": "Shanghai Composite (China)", "tipo": "Emergente"},
    "^AXJO": {"nome": "ASX 200 (Austrália)", "tipo": "Desenvolvida"},
    "^MXX": {"nome": "IPC México", "tipo": "Emergente"},
    "^IPSA": {"nome": "IPSA Chile", "tipo": "Emergente"},
    "^MERV": {"nome": "MERVAL Argentina", "tipo": "Emergente"},
    "^IBEX": {"nome": "IBEX 35 Espanha", "tipo": "Emergente"},
    "^KLSE": {"nome": "KLCI Malásia", "tipo": "Emergente"},
    "^NSEI": {"nome": "Nifty 50 Índia", "tipo": "Emergente"},
    "^SET.BK": {"nome": "SET Tailândia", "tipo": "Emergente"},
    "^JKSE": {"nome": "Jakarta Composite Indonésia", "tipo": "Emergente"},
    "^KS11": {"nome": "KOSPI Coreia do Sul", "tipo": "Emergente"},
    "^STI": {"nome": "Straits Times Singapura", "tipo": "Emergente"}
}

# =====================
# BAIXAR DADOS HISTÓRICOS
# =====================
series_list = []
ticker_name_map = {}  # mapear nome amigável para ticker
start_date = "2010-01-01"

for key, info in tickers.items():
    df_temp = yf.download(key, start=start_date, progress=False, interval='1mo')
    if isinstance(df_temp, pd.DataFrame) and "Close" in df_temp.columns:
        s = df_temp["Close"].copy()
    elif isinstance(df_temp, pd.Series):
        s = df_temp.copy()
    else:
        try:
            s = df_temp.iloc[:, 0].copy()
        except Exception:
            continue
    friendly_name = f"{info['nome']} ({info['tipo']})"
    s.name = friendly_name
    series_list.append(s)
    ticker_name_map[friendly_name] = key  # salvar referência do ticker original

# =====================
# UNIR TODAS EM UM DATAFRAME
# =====================
df_all = pd.concat(series_list, axis=1).dropna(how="all").ffill().dropna(axis=1, how="all")
df_all.columns = [str(c) for c in df_all.columns]

# =====================
# NORMALIZAR BASE 100
# =====================
df_norm = df_all.div(df_all.iloc[0]).mul(100)


# =====================
# CONFIGURAÇÃO DO DASHBOARD
# =====================
num_colunas = 5
cols = list(ticker_name_map.items())
num_plots = len(cols)
num_linhas = (num_plots + num_colunas - 1) // num_colunas

# Subplot titles com ticker original usando map seguro
subplot_titles = [f"{col[0]} — {col[1]}" for col in cols]

fig = make_subplots(
    rows=num_linhas,
    cols=num_colunas,
    subplot_titles=subplot_titles,
    shared_xaxes=False,
    shared_yaxes=False,
    vertical_spacing=0.10,
    horizontal_spacing=0.05
)

# Adicionar cada série em seu subplot
for idx, col_name in enumerate(cols):
    row = idx // num_colunas + 1
    col = idx % num_colunas + 1

    fig.add_trace(
        go.Scatter(
            x=df_norm.index,
            y=df_norm.iloc[:, idx],
            mode="lines",
            hovertemplate="%{x|%Y-%m-%d}<br>Valor Base 100: %{y:.2f}<extra></extra>",
            line=dict(width=2)
        ),
        row=row,
        col=col
    )

# =====================
# DIMINUIR FONTE DOS TÍTULOS DOS SUBPLOTS
# =====================
# Ajuste da fonte dos títulos individuais
for i in range(num_plots):
    fig.layout.annotations[i].font.size = 10  # diminui tamanho da fonte (ex: 10)

# Layout global
fig.update_layout(
    template="plotly_white",
    height=650*num_linhas/2,  # ajusta altura proporcional ao número de linhas
    title_text="Histórico Normalizado das Bolsas Globais",
    title_x=0.5,
    showlegend=False
)
# Ativar grid para todos os eixos
for i in range(num_plots):
    ai_row = i // num_colunas + 1
    ai_col = i % num_colunas + 1
    fig.update_xaxes(showgrid=True, row=ai_row, col=ai_col)
    fig.update_yaxes(showgrid=True, row=ai_row, col=ai_col)

fig.show()
